# Random Forest with NLP (TF-IDF)

Uses preprocessing_NLP.py and combines numeric + text features.

In [1]:
import os
import numpy as np
import pandas as pd

from preprocessing_NLP import build_raw_dataset_nlp
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    )
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from scipy import sparse as sp
import joblib

## Configuration

In [2]:
DATASETS = {
    "genuine_accounts.csv": 0,
    "fake_followers.csv": 1,
    "social_spambots_1.csv": 1,
    "social_spambots_2.csv": 1,
    "social_spambots_3.csv": 1,
    "traditional_spambots_1.csv": 1,
    "traditional_spambots_2.csv": 1,
    "traditional_spambots_3.csv": 1,
    "traditional_spambots_4.csv": 1,
}

BASE_DIR_CANDIDATES = [
    "datasets_full.csv/datasets_full.csv",
    "./datasets_full.csv/datasets_full.csv",
    "../datasets_full.csv/datasets_full.csv",
    "./data",
]
BASE_DIR = next((p for p in BASE_DIR_CANDIDATES if os.path.isdir(p)), BASE_DIR_CANDIDATES[0])

TWEETS_FILE = "tweets.csv"
TEXT_COLUMN = "text"

USER_FEATURES = [
    "statuses_count", "followers_count", "friends_count",
    "favourites_count", "listed_count", "default_profile",
    "default_profile_image", "verified"
]

TWEET_FEATURES = ["reply_count", "favorite_count", "num_urls", "num_mentions"]

TFIDF_PARAMS = {
    "max_features": 2000,
    "stop_words": "english",
    "sublinear_tf": True,
}

RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1,
}

print("Resolved BASE_DIR:", BASE_DIR)
print("Configured datasets:", len(DATASETS))
print("TF-IDF params:", TFIDF_PARAMS)
print("RF params:", RF_PARAMS)

Resolved BASE_DIR: datasets_full.csv/datasets_full.csv
Configured datasets: 9
TF-IDF params: {'max_features': 2000, 'stop_words': 'english', 'sublinear_tf': True}
RF params: {'n_estimators': 300, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}


## Load Data

In [3]:
raw = build_raw_dataset_nlp(DATASETS, BASE_DIR, TWEET_FEATURES, TWEETS_FILE)
print("Raw data shape:", raw.shape)
raw.head()

Loaded genuine_accounts.csv: 3,474 rows
Loaded fake_followers.csv: 3,351 rows
Loaded social_spambots_1.csv: 991 rows
Loaded social_spambots_2.csv: 3,457 rows
Loaded social_spambots_3.csv: 464 rows
Loaded traditional_spambots_1.csv: 1,000 rows
Loaded traditional_spambots_2.csv: 100 rows
Loaded traditional_spambots_3.csv: 403 rows
Loaded traditional_spambots_4.csv: 1,128 rows

Total rows loaded: 14,368
Raw data shape: (14368, 50)


,id,name,screen_name,statuses_count,followers_count,friends_count,favourites_count,listed_count,url,lang,...,test_set_1,test_set_2,label,source_file,user_id,reply_count,favorite_count,num_urls,num_mentions,text
0,1502026416,TASUKU HAYAKAWA,0918Bask,2177,208,332,265,1,NaN,ja,...,0.0,0.0,0,genuine_accounts.csv,NaN,NaN,NaN,NaN,NaN,NaN
1,2492782375,ro_or,1120Roll,2660,330,485,3972,5,NaN,ja,...,0.0,0.0,0,genuine_accounts.csv,NaN,NaN,NaN,NaN,NaN,NaN
2,293212315,bearclaw,14KBBrown,1254,166,177,1185,0,NaN,en,...,0.0,0.0,0,genuine_accounts.csv,NaN,NaN,NaN,NaN,NaN,NaN
3,191839658,pocahontas farida,wadespeters,202968,2248,981,60304,101,http://t.co/rGV0HIJGsu,en,...,0.0,0.0,0,genuine_accounts.csv,NaN,NaN,NaN,NaN,NaN,NaN
4,3020965143,Ms Kathy,191a5bd05da04dc,82,21,79,5,0,NaN,en,...,0.0,0.0,0,genuine_accounts.csv,NaN,NaN,NaN,NaN,NaN,NaN


## Prepare Features and Split

In [4]:
available_user = [c for c in USER_FEATURES if c in raw.columns]
available_tweet = [c for c in TWEET_FEATURES if c in raw.columns]
if not available_user and not available_tweet:
    raise ValueError("No configured numeric feature columns found in raw data.")

numeric_features = available_user + available_tweet
X_numeric = raw[numeric_features].apply(pd.to_numeric, errors="coerce").fillna(0)

text_series = raw[TEXT_COLUMN].fillna("").astype(str) if TEXT_COLUMN in raw.columns else pd.Series([""] * len(raw))
y = raw["label"].astype(int)

X_train_num, X_test_num, text_train, text_test, y_train, y_test = train_test_split(
    X_numeric,
    text_series,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
class_ratio = neg_count / pos_count if pos_count else 1.0

print("Numeric features:", numeric_features)
print("Train rows:", len(X_train_num), "| Test rows:", len(X_test_num))
print("Class balance:\n", y.value_counts(dropna=False))
print(f"Train class ratio (neg/pos): {class_ratio:.4f}")

Numeric features: ['statuses_count', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'default_profile', 'default_profile_image', 'verified', 'reply_count', 'favorite_count', 'num_urls', 'num_mentions']
Train rows: 11494 | Test rows: 2874
Class balance:
 label
1    10894
0     3474
Name: count, dtype: int64
Train class ratio (neg/pos): 0.3189


## Cross-Validated Benchmark

In [5]:
cv_numeric_features = available_user + available_tweet
cv_text_features = [TEXT_COLUMN] if TEXT_COLUMN in raw.columns else []

if not cv_numeric_features and not cv_text_features:
    raise ValueError("No usable features found for cross-validation.")

cv_input = raw[cv_numeric_features + cv_text_features].copy()
for column in cv_numeric_features:
    cv_input[column] = pd.to_numeric(cv_input[column], errors="coerce").fillna(0)
if cv_text_features:
    cv_input[TEXT_COLUMN] = cv_input[TEXT_COLUMN].fillna("").astype(str)

cv_scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1_macro": "f1_macro",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_transformers = [("num", "passthrough", cv_numeric_features)]
if cv_text_features:
    cv_transformers.append(("text", TfidfVectorizer(**TFIDF_PARAMS), TEXT_COLUMN))

cv_preprocessor = ColumnTransformer(transformers=cv_transformers, remainder="drop")
cv_estimator = Pipeline(
    [
        ("preprocessor", cv_preprocessor),
        ("model", RandomForestClassifier(**RF_PARAMS)),
    ]
)

baseline_cv = DummyClassifier(strategy="most_frequent")
baseline_cv_scores = cross_validate(
    baseline_cv,
    cv_input,
    y,
    cv=cv,
    scoring=cv_scoring,
    n_jobs=-1,
    return_train_score=False,
)

rf_cv_scores = cross_validate(
    cv_estimator,
    cv_input,
    y,
    cv=cv,
    scoring=cv_scoring,
    n_jobs=-1,
    return_train_score=False,
)

print("\n5-fold stratified cross-validation")
for metric_name in cv_scoring:
    baseline_values = baseline_cv_scores[f"test_{metric_name}"]
    model_values = rf_cv_scores[f"test_{metric_name}"]
    print(
        f"{metric_name:>18}: baseline {baseline_values.mean():.4f} ± {baseline_values.std():.4f} | "
        f"random forest {model_values.mean():.4f} ± {model_values.std():.4f}"
    )


5-fold stratified cross-validation
          accuracy: baseline 0.7582 ± 0.0001 | random forest 0.9953 ± 0.0011
 balanced_accuracy: baseline 0.5000 ± 0.0000 | random forest 0.9936 ± 0.0011
          f1_macro: baseline 0.4312 ± 0.0000 | random forest 0.9936 ± 0.0015
           roc_auc: baseline 0.5000 ± 0.0000 | random forest 0.9998 ± 0.0001
 average_precision: baseline 0.7582 ± 0.0001 | random forest 0.9999 ± 0.0000


## TF-IDF (Train Only)

In [6]:
tfidf = TfidfVectorizer(**TFIDF_PARAMS)
X_train_text = tfidf.fit_transform(text_train)
X_test_text = tfidf.transform(text_test)

tfidf_feature_names = [f"tfidf_{t}" for t in tfidf.get_feature_names_out()]
print("TF-IDF train shape:", X_train_text.shape)
print("TF-IDF test shape:", X_test_text.shape)

TF-IDF train shape: (11494, 2000)
TF-IDF test shape: (2874, 2000)


## Combine Numeric + Text

In [7]:
X_train_final = sp.hstack([sp.csr_matrix(X_train_num.values), X_train_text], format="csr")
X_test_final = sp.hstack([sp.csr_matrix(X_test_num.values), X_test_text], format="csr")

all_feature_names = list(X_train_num.columns) + tfidf_feature_names

print("Combined train shape:", X_train_final.shape)
print("Combined test shape:", X_test_final.shape)

Combined train shape: (11494, 2012)
Combined test shape: (2874, 2012)


## Train

In [8]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_final, y_train)

model = RandomForestClassifier(**RF_PARAMS)
model.fit(X_train_final, y_train)

print("Training complete.")
print("Number of trees:", len(model.estimators_))

Training complete.
Number of trees: 300


## Evaluate

In [9]:
y_pred = model.predict(X_test_final)
y_proba = model.predict_proba(X_test_final)[:, 1]

baseline_pred = baseline.predict(X_test_final)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "f1_macro": f1_score(y_test, y_pred, average="macro"),
    "roc_auc": roc_auc_score(y_test, y_proba),
    "average_precision": average_precision_score(y_test, y_proba),
}

print("Holdout metrics (Random Forest + NLP)")
for metric_name, metric_value in metrics.items():
    print(f"{metric_name:>18}: {metric_value:.4f}")

print("\nBaseline on same holdout")
print(f"{'accuracy':>18}: {accuracy_score(y_test, baseline_pred):.4f}")
print(f"{'balanced_accuracy':>18}: {balanced_accuracy_score(y_test, baseline_pred):.4f}")
print(f"{'f1_macro':>18}: {f1_score(y_test, baseline_pred, average='macro'):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

Holdout metrics (Random Forest + NLP)
          accuracy: 0.9948
 balanced_accuracy: 0.9917
          f1_macro: 0.9929
           roc_auc: 0.9998
 average_precision: 0.9999

Baseline on same holdout
          accuracy: 0.7582
 balanced_accuracy: 0.5000
          f1_macro: 0.4312

Confusion Matrix:
[[ 685   10]
 [   5 2174]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9928    0.9856    0.9892       695
           1     0.9954    0.9977    0.9966      2179

    accuracy                         0.9948      2874
   macro avg     0.9941    0.9917    0.9929      2874
weighted avg     0.9948    0.9948    0.9948      2874



## Feature Importance and Save

In [10]:
importance_df = pd.DataFrame({
    "feature": all_feature_names,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

print("Top 20 features by importance:")
display(importance_df.head(20))

MODEL_PATH = "random_forest_bot_detector_nlp.joblib"
joblib.dump({
    "model": model,
    "tfidf": tfidf,
    "numeric_features": list(X_train_num.columns),
    "text_column": TEXT_COLUMN,
}, MODEL_PATH)
print(f"Saved NLP bundle to {MODEL_PATH}")

Top 20 features by importance:


,feature,importance
3,favourites_count,0.254261
0,statuses_count,0.158530
1,followers_count,0.076592
11,num_mentions,0.056668
9,favorite_count,0.045597
10,num_urls,0.042970
2,friends_count,0.041750
4,listed_count,0.027643
806,tfidf_http,0.017644
8,reply_count,0.012786


Saved NLP bundle to random_forest_bot_detector_nlp.joblib
